In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
df_train = pd.read_csv("train_clean.csv")
df_val   = pd.read_csv("val_clean.csv")
df_test  = pd.read_csv("test_clean.csv")

print(df_train.head())

                                                text  label
0  प्रेमिका को बर्थडे विश जेल से भागा था अंकित धन...      1
1  जो भी भाई ०प जॉब वाले गाँव आ रहे है कृपा करके ...      2
2  माँ घबराते हुए बोली बेटा जल्दी घर आ जाओ बहु को...      1
3  मेरे सिंगल भाईयों तुम लोग कहो तो १ फरवरी को भज...      2
4  एक तो मे मासूम ओर ऊपर से शरीफ मुष्मे न्ो बिगाड़...      1


In [3]:
train_ds = Dataset.from_pandas(df_train)
val_ds   = Dataset.from_pandas(df_val)
test_ds  = Dataset.from_pandas(df_test)

In [4]:
model_name = "ai4bharat/indic-bert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/8400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [5]:
num_labels = 3  # positive, negative, neutral

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_indicbert",
    eval_strategy="epoch",      # run eval after every epoch
    save_strategy="epoch",            # save checkpoint each epoch
    save_total_limit=2,               # keep only best 2 checkpoints
    learning_rate=3e-5,               # slightly higher LR than default
    per_device_train_batch_size=32,   # bigger batch (your GPU can handle)
    per_device_eval_batch_size=32,
    num_train_epochs=6,               # train longer (try 6–8)
    weight_decay=0.01,
    warmup_ratio=0.1,                 # LR warmup (stabilizes training)
    load_best_model_at_end=True,      # auto-pick best checkpoint
    metric_for_best_model="f1",       # compare models using F1
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",                 # disable wandb/hub
    fp16=True,                        # use mixed precision on NVIDIA GPU
)
print("🚀 TrainingArguments optimized for high accuracy")


🚀 TrainingArguments optimized for high accuracy


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\nakul\AppData\Local\Temp\ipykernel_13052\3533869841.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.035100,1.010782,0.441667,0.479677,0.441667,0.371290
2,0.991200,0.954082,0.513889,0.566287,0.513889,0.510474
3,0.905600,0.915928,0.532222,0.619661,0.532222,0.530923
4,0.825700,0.895513,0.555556,0.633497,0.555556,0.554454
5,0.722700,0.971078,0.555000,0.582623,0.555000,0.557924
6,0.567900,1.085627,0.555556,0.574911,0.555556,0.557860


TrainOutput(global_step=1578, training_loss=0.8500703961525882, metrics={'train_runtime': 559.6165, 'train_samples_per_second': 90.062, 'train_steps_per_second': 2.82, 'total_flos': 301145848012800.0, 'train_loss': 0.8500703961525882, 'epoch': 6.0})

In [14]:
metrics = trainer.evaluate(test_ds)
print(metrics)

{'eval_loss': 0.9569858312606812, 'eval_accuracy': 0.5683333333333334, 'eval_precision': 0.590782814110842, 'eval_recall': 0.5683333333333334, 'eval_f1': 0.5704932731427387, 'eval_runtime': 7.3751, 'eval_samples_per_second': 244.064, 'eval_steps_per_second': 7.729, 'epoch': 6.0}


In [15]:
trainer.save_model("./best_hindi_bert")
tokenizer.save_pretrained("./best_hindi_bert")

('./best_hindi_bert\\tokenizer_config.json',
 './best_hindi_bert\\special_tokens_map.json',
 './best_hindi_bert\\spiece.model',
 './best_hindi_bert\\added_tokens.json',
 './best_hindi_bert\\tokenizer.json')